# Setup and Load Data

In [0]:
from pyspark.sql.functions import (
    to_timestamp, substring, lpad, concat,
    year, month, dayofmonth, make_timestamp, 
    col, count, when, isnan, avg, sum as spark_sum, lit, desc,
    min, max, mean, stddev, log1p, broadcast,
    regexp_replace, try_to_timestamp, date_trunc, expr, least
)
# from pyspark.sql.functions import to_timestamp# , col, date_trunc, hour, expr, regexp_replace, try_to_timestamp
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import DataFrame
from graphframes import GraphFrame
from functools import reduce

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
# # Checkpoint as parquet (run once, then comment out)
# TEAM_DIR = "dbfs:/student-groups/Group_3_2"
# dbutils.fs.mkdirs(TEAM_DIR)
# df_otpw.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_3m.parquet")
# print("Parquet saved.")

# Read in 

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
#df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
otpw_full_df = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")
weather_full = spark.read.parquet('dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/')

# Rejoin OTPW and Weather DF for weather to be matched with flight 2 hours before departure (to prevent leakage)

In [0]:
def rejoin_otpw_and_weather(otpw_df, weather_df):
    '''Rejoin OTPW and Weather DF such that recorded weather observations are at least 2 hours before a flight's scheduled departure'''
    # filter on fm-15
    df_weather_filtered = weather_df.filter(col('REPORT_TYPE') == 'FM-15')   

    stations_set = {
        row["origin_station_id"]
        for row in otpw_df.select("origin_station_id").distinct().collect()
    }

    # filter on only stations in otpw
    df_weather_filtered = df_weather_filtered.filter(col('station').isin(stations_set))
    
    # Convert to timestamp. Create "ts" from "DATE"
    df_weather_filtered = df_weather_filtered.withColumn(
        "ts",
        to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss")
    )

    df_weather_filtered = df_weather_filtered.filter(col("ts").isNotNull())

    # Truncate to hour. Create hour_trunc from "ts"
    df_weather_filtered = df_weather_filtered.withColumn(
        "hour_trunc",
        date_trunc("hour", col("ts"))
    )

    # List of columns to keep
    cols_to_keep = [
        'STATION',
        'DATE',
        'REPORT_TYPE',
        'SOURCE',
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyRelativeHumidity',
        'HourlyVisibility',
        'HourlyWindSpeed',
        'hour_trunc'
    ]

    # Filter the DataFrame to only these columns
    df_weather_filtered = df_weather_filtered.select(*cols_to_keep)

    # In df_weather_filtered, rename STATION → origin_station_id to match OTPW
    df_weather_filtered = df_weather_filtered.withColumnRenamed("STATION", "origin_station_id")

    weather_cols = [
        "HourlyDryBulbTemperature",
        "HourlyDewPointTemperature",
        "HourlyWindSpeed",
        "HourlyWindGustSpeed",
        "HourlyWindDirection",
        "HourlyVisibility",
        "HourlyPrecipitation",
        "HourlyRelativeHumidity",
        "HourlyStationPressure",
        "HourlySeaLevelPressure"
    ]

    df_weather_renamed = df_weather_filtered
    for col_name in weather_cols:
        df_weather_renamed = df_weather_renamed.withColumnRenamed(col_name, f"2h_{col_name}")

    # drop duplicates
    df_weather_renamed = df_weather_renamed.dropDuplicates(["origin_station_id", "hour_trunc"])

    # Drop columns that are no longer needed (original hourly weather vars and DATE)
    df_weather_renamed = df_weather_renamed.drop(*weather_cols)
    df_weather_renamed = df_weather_renamed.drop('DATE')
    otpw_df = otpw_df.drop(col('REPORT_TYPE'), col('SOURCE'))

    # Create "sched_depart_date_time" column, the scheduled flight departure in local time.
    otpw_df = otpw_df.withColumn(
        "sched_depart_date_time",
        make_timestamp(
            col("YEAR"),             # year
            col("MONTH"),         # month
            col("DAY_OF_MONTH"),        # day
            substring(lpad(col("CRS_DEP_TIME"), 4, "0"), 1, 2),  # hour
            substring(lpad(col("CRS_DEP_TIME"), 4, "0"), 3, 2),  # minute
            col("sched_depart_date_time_UTC").substr(15, 2)      # seconds (usually "00")
        )
    )

    # Create "actual_depart_date_time" column, the actual flight departure in local time.
    otpw_df = otpw_df.withColumn(
        "actual_depart_date_time",
        make_timestamp(
            col("YEAR"),             # year
            col("MONTH"),         # month
            col("DAY_OF_MONTH"),        # day
            substring(lpad(col("DEP_TIME"), 4, "0"), 1, 2),  # hour
            substring(lpad(col("DEP_TIME"), 4, "0"), 3, 2),  # minute
            lit("00")      # seconds (usually "00")
        )
    )

    # otpw_df = otpw_df.withColumn(
    #     "sched_depart_date_time",
    #     regexp_replace(col("sched_depart_date_time"), r'T(\d):', r'T0\1:')
    # )

    # otpw_df = otpw_df.withColumn(
    #     "sched_depart_date_time",
    #     regexp_replace(col("sched_depart_date_time"), r':(\d):', r':0\1:')
    # )

    # Optional safety
    otpw_df = otpw_df.filter(col("sched_depart_date_time").isNotNull())
    otpw_df = otpw_df.filter(col("actual_depart_date_time").isNotNull())
    otpw_df = otpw_df.withColumn("hour_trunc", date_trunc("hour", least(col("sched_depart_date_time"), col("actual_depart_date_time")) - expr("INTERVAL 2 HOURS")))

    otpw_df_2h = otpw_df.join(
        df_weather_renamed,
        on=["origin_station_id", "hour_trunc"],  # join keys
        how="left"
    )

    return otpw_df_2h

def get_dimensions(df):
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")

In [0]:
otpw_full_2h = rejoin_otpw_and_weather(otpw_full_df, weather_full)

get_dimensions(otpw_full_df)
get_dimensions(otpw_full_2h)

There are 470k (1.5%) less rows in the joined dataset due to having to filter out null values for the actual departing date time. This is likely due to flights never departing due to cancellation. This should not be too much of an issue as we still have over 30 million rows to work with and we avoid the issue of data leakage as we want our weather measurements to be recorded at least 2 hours before a flight's scheduled or actual departure, whichever comes earlier.

In [0]:
display(otpw_full_2h)

In [0]:
# Ensure no duplicate columns
import collections
duplicate_cols = [item for item, count in collections.Counter(otpw_full_2h.columns).items() if count > 1]
print(f"Duplicate columns: {duplicate_cols}")

# Split into train and test sets before conducting feature engineering to prevent data leakage

In [0]:
def otpw_train_test_split(otpw_df):
    '''Split otpw data into train (2015-2018) and blind test (2019) sets'''

    train_df = otpw_df.filter(col("sched_depart_date_time").between("2015-01-01", "2018-12-31"))
    test_df = otpw_df.filter(col("sched_depart_date_time").between("2019-01-01", "2019-12-31"))
    return train_df, test_df
    
otpw_train_df, otpw_test_df = otpw_train_test_split(otpw_full_2h)

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041126_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041126_otpw_test_df.parquet")

# Verify the min and max dates for each split

In [0]:
otpw_train_df.select(
    min("sched_depart_date_time").alias('min_date'),
    max("sched_depart_date_time").alias('max_date')
).show()

otpw_test_df.select(
    min("sched_depart_date_time").alias('min_date'),
    max("sched_depart_date_time").alias('max_date')
).show()

# Correlation (on the training set from 2015-2018)

In [0]:
columns = [
    "DEP_DEL15",
    "DISTANCE",
    "CRS_ELAPSED_TIME",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyWindDirection",
    "HourlyVisibility",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyStationPressure",
    "HourlySeaLevelPressure"
]

# df_casted = df.select([col(c).cast("double").alias(c) for c in columns])
otpw_train_df_casted = otpw_train_df.select([col(c).cast("double").alias(c) for c in columns])


In [0]:
from pyspark.sql.functions import when

# df_cleaned = df.select([
#     when(col(c).isin("", "NA", "null"), None)
#     .otherwise(col(c))
#     .cast("double")
#     .alias(c)
#     for c in columns
# ]).dropna()
otpw_train_df_cleaned = otpw_train_df.select([
    when(col(c).isin("", "NA", "null"), None)
    .otherwise(col(c))
    .cast("double")
    .alias(c)
    for c in columns
]).dropna()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

assembler = VectorAssembler(inputCols=columns, outputCol="features")
# df_vector = assembler.transform(df_cleaned)
df_vector = assembler.transform(otpw_train_df_cleaned)

corr_matrix = Correlation.corr(df_vector, "features", method="spearman").head()[0]

corr_matrix.toArray()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

corr_array = corr_matrix.toArray()
corr_df = pd.DataFrame(corr_array, index=columns, columns=columns)

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5
)
plt.title("Spearman Correlation Heatmap")
plt.show()

# General Info and Feature Types

In [0]:
# Dataset dimensions
num_rows = otpw_train_df.count()
num_cols = len(otpw_train_df.columns)
print(f"Rows: {num_rows:,}")
print(f"Columns: {num_cols}")

# Handle Duplicates and Missing Data

## Duplicates

In [0]:
total = otpw_train_df.count()
distinct = otpw_train_df.dropDuplicates().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicate rows: {total - distinct}")

## Missing Data for Response Variable

In [0]:
null_exprs = [
    count(
        when(col(c).isNull() | (col(c).cast("string") == ""), c)
    ).alias(c)
    for c in otpw_train_df.columns
]
null_counts = otpw_train_df.select(null_exprs).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_percentage"] = (null_counts["null_count"] / num_rows * 100).round(2)

null_counts = (
    null_counts
    .reset_index()
    .rename(columns={"index": "column_name"})
    .sort_values("null_percentage", ascending=False)
)

high_missing_cols = null_counts[null_counts["null_percentage"] > 50]
print(f"Columns with >50% missing: {len(high_missing_cols)}")
display(null_counts)

In [0]:
delayed = otpw_train_df.filter(col("DEP_DEL15") == 1)
on_time = otpw_train_df.filter(col("DEP_DEL15") == 0)
missing = otpw_train_df.filter(col("DEP_DEL15").isNull())

print(f"Total delayed flights: {delayed.count()}")
print(f"Total on-time flights: {on_time.count()}")
print(f"Missing DEP_DEL15: {missing.count()}")
print(f"Distinct flights: {distinct}")
print(f"Missing flight percentage (%): {missing.count() / distinct * 100:.2f}")

We can see that DEP_DEL15, the variable we are interested in, has 0.02 percent missing data. This could be due to the fact that there are cancelled flights that may have null DEP_DEL15 values.

In [0]:
# Is it true that all cancelled flights will not have DEP_DELAY and DEP_DEL15?
cancelled = otpw_train_df.filter(col("CANCELLED") == 1)
total_cancelled = cancelled.count()
cancelled_with_delay = cancelled.filter(col('DEP_DELAY').isNotNull()).count()
cancelled_with_del15 = cancelled.filter(col('DEP_DEL15').isNotNull()).count()

print(f"Total cancelled flights: {total_cancelled} ({total_cancelled / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DELAY: {cancelled_with_delay} ({cancelled_with_delay / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DEL15: {cancelled_with_del15} ({cancelled_with_del15 / num_rows * 100:.2f}% of dataset)")

We decide to remove cancelled flights as it may introduce confounding variables on an already unbalanced dataset.

In [0]:
# Filter out cancelled flights
otpw_train_df = otpw_train_df.filter(col("CANCELLED") == 0)
num_rows = otpw_train_df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
missing = otpw_train_df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing.count()}")

There are still 4744 flights with no departure delay label. We can impute these missing values by computing the difference between CRS_DEP_TIME and DEP_TIME.

In [0]:
display(missing)

In [0]:
def impute_missing_target_var(otpw_df):
    # 1. Build full timestamps for scheduled and actual departure times
    otpw_filled = otpw_df.withColumn(
        "sched_ts",
        to_timestamp(
            concat(
                col("FL_DATE"), 
                lpad(col("CRS_DEP_TIME"), 4, "0")
            ),
            "yyyy-MM-ddHHmm"
        )
    ).withColumn(
        "actual_ts",
        to_timestamp(
            concat(
                col("FL_DATE"), 
                lpad(col("DEP_TIME"), 4, "0")
            ),
            "yyyy-MM-ddHHmm"
        )
    )

    # 2. Compute difference in minutes
    otpw_filled = otpw_filled.withColumn(
        "dep_diff_mins",
        (col("actual_ts").cast("long") - col("sched_ts").cast("long")) / 60
    )

    # 3. Fill missing DEP_DEL15 using the computed difference
    otpw_filled = otpw_filled.withColumn(
        "DEP_DEL15",
        when(
            col("DEP_DEL15").isNull(),
            when(col("dep_diff_mins") >= 15, 1).otherwise(0)
        ).otherwise(col("DEP_DEL15")).cast("double")
    )

    return otpw_filled

In [0]:
otpw_train_df = impute_missing_target_var(otpw_train_df)
missing_dep_del15_train = otpw_train_df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing_dep_del15_train.count()}")

We will also removed cancelled flights from the test set and impute the missing target variable "DEP_DEL15" as well. This will not introduce data leakage as cancelled flights and the imputation is independent from all other flights. That is, the flight's overall status and departure depends on the flight's own metadata.

In [0]:
otpw_test_df = otpw_test_df.filter(col("CANCELLED") == 0)
otpw_test_df = impute_missing_target_var(otpw_test_df)
missing_dep_del15_test = otpw_test_df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing_dep_del15_test.count()}")

In the train and test sets, there are no more missing data for DEP_DEL15.

In [0]:
# Class balance of DEP_DEL15
target_dist = otpw_train_df.groupBy("DEP_DEL15").count().toPandas()
target_dist["percentage"] = (target_dist["count"] / target_dist["count"].sum() * 100).round(2)
print(target_dist)

plt.figure(figsize=(6, 4))
plt.bar(target_dist["DEP_DEL15"].astype(str), target_dist["count"])
plt.title("DEP_DEL15 Class Distribution (2015-2018 OTPW)")
plt.xlabel("DEP_DEL15 (0=On-time, 1=Delayed ≥15min)")
plt.ylabel("Count")
for i, row in target_dist.iterrows():
    plt.text(i, row["count"], f"{row['percentage']}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Other missing data explaination

A significant amount of missing data comes from the weather categories. However, we would like to use this data since we believe weather is an important predictor to weather or not a flight is delayed. Specifically, we are concerned with hourly weather categories.

We first check report type for missing data of weather. One of the reasons why there are so many missing values is that the weather dataset has multiple report types (FM-15, FM-16, FM-12, etc) that do not contain all information. For example:

* FM-15: Standard hourly temperature, wind, visibility, pressure, and precipitation observations. This is the most complete and consistent report type, making it ideal for linking to individual flights.
* FM-16: Special weather report issued when conditions change significantly (e.g., sudden storms or extreme winds). It is irregular and event-driven, so coverage is sparse.
* FM-12: Synoptic report issued every 6–12 hours with fewer variables; many fields are missing and it does not align well with hourly flight data.
* SOD (Summary of Day): Daily aggregates like total precipitation or max/min temperature; not useful for per-flight modeling.
* SHEF and SY-MT: Rare or specialized reports (hydrology or system metadata) with minimal rows and missing data.

Because of this, we focus primarily on FM-15 for modeling. In the future we can optionally include FM-16 if we want to capture extreme weather events. As for the other report tyes, we will and drop them to ensure consistent and reliable features for each flight.

In [0]:
otpw_train_df.groupBy("REPORT_TYPE") \
  .agg(count("*").alias("count")) \
  .display()

In [0]:
otpw_train_df = otpw_train_df.filter(col("REPORT_TYPE") == "FM-15")
num_rows = otpw_train_df.count()
print(f"After filtering FM-15: {num_rows:,} rows remaining")

Next, we look into source type, since the weather dataset is based off of many different sources, there may be sources that have more missing data than others.

In [0]:
otpw_train_df.groupBy("SOURCE").count().display()

In [0]:
weather_cols = [
    "HourlyAltimeterSetting",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyPresentWeatherType",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlyRelativeHumidity",
    "HourlySkyConditions",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlyVisibility",
    "HourlyWetBulbTemperature",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWindSpeed"
]

exprs = [
    (count(when(col(c).isNull(), True)) / count("*")).alias(f"{c}_missing_pct")
    for c in weather_cols
]

otpw_train_df.groupBy("SOURCE").agg(*exprs).display()

In the table, we can see that source 7 and 6 make up the majority of the dataset. There are missing data, but there are also a lot of columns that have are filled out in the majority of rows.

Additionally based off of the data dictionary to the original data, null values represent zero or not observed, and M represetns missing data. 

In the OTPW dataset, there are both 0 and null values in precipitation, but we believe that is just an artifact of merging the data. Specifically when the report type is FM-15 (regular at hourly intervals), we should treat nulls as 0s.

In [0]:
from pyspark.sql.types import FloatType

weather_continuous_cols = [
    "HourlyAltimeterSetting",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    #"HourlyPresentWeatherType",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlyRelativeHumidity",
    #"HourlySkyConditions",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlyVisibility",
    "HourlyWetBulbTemperature",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWindSpeed"
]

for c in weather_cols:
    otpw_train_df = otpw_train_df.withColumn(c, col(c).cast(FloatType()))
    if  c in {"HourlyPrecipitation"}:
        otpw_train_df = otpw_train_df.na.fill({c: 0})  # fill nulls column by column

In [0]:
otpw_train_df.display(100)

# Show percentage of null values for weather columns

In [0]:
def get_null_info(df):
    '''Display number of null records and null percentage for each column in the dataframe'''
    total_rows = df.count()

    exprs = [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    counts = df.select(exprs)

    # Convert wide → long
    long_df = counts.select(
        F.explode(F.map_from_arrays(
            F.array(*[F.lit(c) for c in df.columns]),
            F.array(*[F.col(c) for c in df.columns])
        )).alias("column", "null_count")
    )

    # Add percentage
    return long_df.withColumn(
        "null_pct", 
        F.concat(
            F.round((F.col("null_count") / total_rows)*100, 2).cast("string"),
            F.lit("%")
        )
    )

In [0]:
otpw_train_df_weather_null_info = get_null_info(otpw_train_df.select(weather_continuous_cols))
display(otpw_train_df_weather_null_info)

# Standardization and Transformations

Based off of the table above showing the missingness of our weather data, we drop `HourlyWindGustSpeed`, `HourlyPressureChange`, `HourlyPressureTendency` due to severe missingness (>50%). We also remove, `HourlyAltimeterSetting`, `HourlySeaLevelPressure`, and `HourlyStationPressure` because they are unlikely to provide any predictive value toward delays. Since `HourlyWetBulbTemperature` and `HourlyDewPointTemperature` are highly correlated with `HourlyDryBulbTemperature` (which is primarily used as the standard air temperature
reported), we keep only `HourlyDryBulbTemperature`. Finally, because `HourlyWindDirection` is correlated with `HourlyWindSpeed`, the latter of which might be more relevant towards flight delays, we drop `HourlyWindDirection`.

This leaves us with `HourlyPrecipitation`, `HourlyRelativeHumidity`, `HourlyWindSpeed`, `HourlyVisibility` and `HourlyDryBulbTemperature` as our flight delay signals. We will use them in our modeling, and they will need to be standardized and transformed.

In [0]:
# Collect data to pandas (small enough sample!)
precip_pd = otpw_train_df.select("HourlyPrecipitation").sample(0.1).toPandas()  # sample 10% to speed up

plt.figure(figsize=(8,5))
plt.hist(precip_pd['HourlyPrecipitation'], bins=50)
plt.xlabel('Hourly Precipitation')
plt.ylabel('Count')
plt.title('Distribution of Hourly Precipitation')

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()

In [0]:
import numpy as np
from pyspark.sql.functions import log1p

plt.hist(precip_pd['HourlyPrecipitation'].apply(lambda x: np.log1p(x)), bins=50)
plt.xlabel('Log(Precipitation + 1)')
plt.ylabel('Count')
plt.title('Log-transformed Hourly Precipitation')

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()

Precipitation is heavily skewed, even after log transformations it is still skewed. A better option would be to turn it into a binary feature where 0 = no precipitation and 1 = precipitation

In [0]:
otpw_train_df = otpw_train_df.withColumn(
    "PrecipBinary",
    when(col("HourlyPrecipitation") > 0, 1).otherwise(0)
)

otpw_test_df = otpw_test_df.withColumn(
    "PrecipBinary",
    when(col("HourlyPrecipitation") > 0, 1).otherwise(0)
)

In [0]:
# Sample data to pandas (optional for large datasets)
sample_pd = otpw_train_df.select("PrecipBinary").toPandas()

# Count occurrences of 0 and 1
counts = sample_pd["PrecipBinary"].value_counts().sort_index()

# Compute percentages
percentages = counts / counts.sum() * 100

# Plot bar chart
plt.figure(figsize=(6,4))
bars = plt.bar(counts.index.astype(str), counts.values)

plt.xticks([0,1])
plt.xlabel("Precipitation Binary (0 = No Rain, 1 = Any Rain)")
plt.ylabel("Count")
plt.title("Distribution of Binary Precipitation")

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

# Add percentage labels on top of bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    pct = percentages.iloc[i]
    ct = counts.iloc[i]
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f"{ct} ({pct:.1f}%)", ha='center', va='bottom')

plt.show()

Next we look at the other numeric features: humidity, wind speed, visibility and temperature.

In [0]:
weather_numeric_cols = [
    "HourlyRelativeHumidity",
    "HourlyWindSpeed",
    "HourlyVisibility",
    "HourlyDryBulbTemperature"
]

sample_pd = otpw_train_df.select(weather_numeric_cols).sample(0.1, seed=42).toPandas()

plt.figure(figsize=(16,10))

for i, col_name in enumerate(weather_numeric_cols):
    ax = plt.subplot(2, 2, i+1)
    ax.hist(sample_pd[col_name], bins=50)
    ax.set_title(f'Distribution of {col_name}')
    ax.set_xlabel(col_name)
    ax.set_ylabel('Count')

    # Disable scientific notation for THIS subplot
    ax.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

Relative humidity and temperature look slightly left-skewed. We will standardize these features.

Wind speed is right skewed. We will log transform it and then standardize it.

Finally, visibility is sparse and left skewed, with a max. We decide to make it into a categorical variable instead.

# Standardize relative humidity and dry bulb temperature

In [0]:
features = ["HourlyRelativeHumidity", "HourlyDryBulbTemperature"]

# Compute stats in one aggregation
agg_exprs = [mean(c).alias(f"{c}_mean") for c in features] + \
            [stddev(c).alias(f"{c}_std") for c in features]

train_stats = otpw_train_df.agg(*agg_exprs).first()

# Apply standardization on train and test set using the train statistics
for c in features:
    train_mu = train_stats[f"{c}_mean"]
    train_sd = train_stats[f"{c}_std"]
    otpw_train_df = otpw_train_df.withColumn(
        f"{c}_stdized", 
        (col(c) - train_mu) / train_sd
    )
    otpw_test_df = otpw_test_df.withColumn(
        f"{c}_stdized",
        (col(c) - train_mu) / train_sd
    )

In [0]:
# Columns to plot
stdized_cols = ["HourlyRelativeHumidity_stdized", "HourlyDryBulbTemperature_stdized"]

sample_pd = otpw_train_df.select(stdized_cols).toPandas()

# Plot histograms
plt.figure(figsize=(12,5))

for i, col_name in enumerate(stdized_cols):
    ax = plt.subplot(1, 2, i+1)
    ax.hist(sample_pd[col_name], bins=50)
    ax.set_title(f'Distribution of {col_name}')
    ax.set_xlabel(col_name)
    ax.set_ylabel('Count')

    # Disable scientific notation for THIS subplot
    ax.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

# Log Transform and standardize Wind Speed

In [0]:
# Log-transform (creates a new column)
otpw_train_df = otpw_train_df.withColumn("HourlyWindSpeed_log", log1p(col("HourlyWindSpeed")))

# Compute mean and std
train_stats = otpw_train_df.select(
    mean("HourlyWindSpeed_log").alias("mean"),
    stddev("HourlyWindSpeed_log").alias("std")
).collect()[0]

# Standardize to mean=0, std=1
otpw_train_df = otpw_train_df.withColumn(
    "HourlyWindSpeed_stdized",
    (col("HourlyWindSpeed_log") - train_stats["mean"]) / train_stats["std"]
)

# Log-transform on test set
otpw_test_df = otpw_test_df.withColumn(
    "HourlyWindSpeed_log",
    log1p(col("HourlyWindSpeed"))
)

# 2. Apply training-set standardization params on test set
otpw_test_df = otpw_test_df.withColumn(
    "HourlyWindSpeed_stdized",
    (col("HourlyWindSpeed_log") - train_stats["mean"]) / train_stats["std"]
)

In [0]:
sample_pd = otpw_train_df.select("HourlyWindSpeed_stdized").toPandas()

# Plot histogram
plt.figure(figsize=(8,5))
plt.hist(sample_pd["HourlyWindSpeed_stdized"], bins=50)
plt.xlabel("HourlyWindSpeed_stdized")
plt.ylabel("Count")
plt.title("Histogram of Standardized Log-Transformed Wind Speed")

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()


# Transform Visibility into a categorical variable with the following categories:
| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

In [0]:
from pyspark.sql.functions import when, col

# Categorize visibility on train set
otpw_train_df = otpw_train_df.withColumn(
    "VisibilityCat",
    when(col("HourlyVisibility") < 2, 0)
    .when((col("HourlyVisibility") >= 2) & (col("HourlyVisibility") < 5), 1)
    .when((col("HourlyVisibility") >= 5) & (col("HourlyVisibility") < 10), 2)
    .otherwise(3)
)

# Categorize visibility on test set. No leakage because test set does not influence learned parameters or use future info.
otpw_test_df = otpw_test_df.withColumn(
    "VisibilityCat",
    when(col("HourlyVisibility") < 2, 0)
    .when((col("HourlyVisibility") >= 2) & (col("HourlyVisibility") < 5), 1)
    .when((col("HourlyVisibility") >= 5) & (col("HourlyVisibility") < 10), 2)
    .otherwise(3)
)

In [0]:
# Sample to Pandas (optional for large datasets)
sample_pd = otpw_train_df.select("VisibilityCat").sample(0.1, seed=42).toPandas()

# Count occurrences of each category
counts = sample_pd["VisibilityCat"].value_counts().sort_index()

# Plot bar chart
plt.figure(figsize=(6,4))
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Visibility Category (0=Very Low, 3=High)")
plt.ylabel("Count")
plt.title("Distribution of Visibility Categories")
plt.xticks(counts.index)

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

plt.show()

In [0]:
# Sample to Pandas (optional for large datasets)
sample_pd = otpw_train_df.select("VisibilityCat").toPandas()

# Count occurrences of each category
counts = sample_pd["VisibilityCat"].value_counts().sort_index()

# Compute percentages
percentages = counts / counts.sum() * 100

# Plot bar chart with counts on y-axis
plt.figure(figsize=(6,4))
bars = plt.bar(counts.index.astype(str), counts.values)

plt.xlabel("Visibility Category (0=Very Low, 1=Low, 2=Moderate, 3=High)")
plt.ylabel("Count")
plt.title("Distribution of Visibility Categories")
plt.xticks(counts.index)

# Disable scientific notation on y-axis
plt.ticklabel_format(style='plain', axis='y')

# Add percentage labels on top of bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    pct = percentages.iloc[i]
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f"{pct:.1f}%", ha='center', va='bottom')

plt.show()

# Feature Engineering and Creation

We will create two additional features, measuring if the flight is on a weekend, as well as and combination of the origin and destination airport.

In [0]:
otpw_train_df = otpw_train_df.withColumn(
    "IsWeekend",
    when((col("DAY_OF_WEEK") == 6) | (col("DAY_OF_WEEK") == 7), 1).otherwise(0)
)

In [0]:
otpw_train_df.groupBy("IsWeekend").count().display()

In [0]:
from pyspark.sql.functions import concat_ws, col

otpw_train_df = otpw_train_df.withColumn(
    "Route",
    concat_ws("_", col("ORIGIN"), col("DEST"))
)

In [0]:
# Perform the same weekend and route feature creations on the test set

otpw_test_df = otpw_test_df.withColumn(
    "IsWeekend",
    when((col("DAY_OF_WEEK") == 6) | (col("DAY_OF_WEEK") == 7), 1).otherwise(0)
)

otpw_test_df = otpw_test_df.withColumn(
    "Route",
    concat_ws("_", col("ORIGIN"), col("DEST"))
)


We will need to encode this new variable for ML models

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Step 1: Convert Route string into numeric index
route_indexer = StringIndexer(inputCol="Route", outputCol="RouteIndex", handleInvalid="keep")
# otpw_train_df = route_indexer.fit(otpw_train_df).transform(otpw_train_df)
route_indexer_model = route_indexer.fit(otpw_train_df)
otpw_train_df = route_indexer_model.transform(otpw_train_df)

# Step 2: One-hot encode the index
route_encoder = OneHotEncoder(inputCols=["RouteIndex"], outputCols=["RouteVec"])
# otpw_train_df = route_encoder.fit(otpw_train_df).transform(otpw_train_df)
route_encoder_model = route_encoder.fit(otpw_train_df)
otpw_train_df = route_encoder_model.transform(otpw_train_df)

In [0]:
# Perform the same one-hot encoding on test set using training-fitted indexer and encoder
otpw_test_df = route_indexer_model.transform(otpw_test_df)
otpw_test_df = route_encoder_model.transform(otpw_test_df)

# Write train and test sets for OTPW to TEAM_DIR

In [0]:
# otpw_train_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041026_otpw_train_df.parquet")
# otpw_test_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041026_otpw_test_df.parquet")

In [0]:
display(dbutils.fs.ls(TEAM_DIR))

# Read in checkpointed train and test data.

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041126_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041126_otpw_test_df.parquet")

# TODO before Saturday 4/11 12pm meeting
Allen: add time-series feature to `otpw_train_df` and `otpw_test_df` while avoiding data leakage (in notebook **Daniel Feature Engineering**)

- it counts and gives the percentage of flights that were delayed at ORIGIN 2-8 hours before the current flight's departure time
- simple idea is that past fights delays probably predict future flight delays well
- its very costly to calculate the variable so we probably want to checkpoint it after

Allen: add graph feature to `otpw_train_df` and `otpw_test_df` while avoiding data leakage

# Daniel: We create a window 2-4 hours before the flight to see the count percentange of delayed flights. 

If there are no flights in that window, it is it counts as 0 (nulls are replaced with 0). We can change the flight window to capture a longer time period before the flight's departure. The windows are grouped by airport (not enough data for it to be grouped by airline in my opinion).

In [0]:
def compute_window_features(otpw_df, lookback_start_hours = -4, lookback_end_hours = -2):
    '''Compute two window-based features from OTPW dataset:
    
        1. The number of delayed flights partitioned by airport in the specified lookback window.

        2. The percentange of delayed flights partitioned by airport in the specified lookback window.

    Default lookback window is 2-4 hours before departure. If there are no flights in the window, we count the # of flights as 0.
    '''
    # Repartition for performance
    otpw_df = otpw_df.repartition("ORIGIN")

    # Define time-based window containing flights based on specified lookback hours before departure
    window_spec = Window.partitionBy("ORIGIN") \
        .orderBy(F.col("sched_depart_date_time").cast("long")) \
        .rangeBetween(lookback_start_hours * 3600, lookback_end_hours * 3600) # can change these number for different window

    # The number of delayed flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"delayed_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.sum("DEP_DEL15").over(window_spec),
            F.lit(0) # replace null with 0
        )
    )

    # The total number of flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"total_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.count("*").over(window_spec),
            F.lit(0)
        )
    )

    # The percentange of delayed flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"delay_pct_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.avg(F.coalesce(F.col("DEP_DEL15"), F.lit(0))).over(window_spec),
            F.lit(0) # replace null with 0
        )
    )

    return otpw_df

In [0]:
otpw_train_df = compute_window_features(otpw_train_df)
otpw_test_df = compute_window_features(otpw_test_df)

We create two additional features above:

1. `delayed_flights_2_4hr_before` gives the number of delayed flights for each airport in the 2-4 hours before this flight.

2. `delay_pct_flights_2_4hr_before` is the percentage of delayed flights for each airport in the same window.

Both features capture recently operational conditions at the airport which we hope carry a predictive signal towards future delays.

However, including delay flight percentage and delayed flight count in logistic regression can result in multicollinearity issues. To avoid multicollinearity for logistic regression, we will use `delay_pct_flights_2_4hr_before` as a normalized measure of congestion, include and log-transform a new feature representing total flight volume `total_flights_2_4hr_before`, and drop `delayed_flights_2_4hr_before`.

In [0]:
# Log transform total flights partitioned by airport
otpw_train_df = otpw_train_df.withColumn(
    "log_total_flights_2_4hr_before",
    log1p(col("total_flights_2_4hr_before"))
)

otpw_test_df = otpw_test_df.withColumn(
    "log_total_flights_2_4hr_before",
    log1p(col("total_flights_2_4hr_before"))
)

# Temporal PageRank by month - Capture Airport network centrality and traffic volume.

In [0]:
# def compute_pagerank_features(df):
#     """
#     Compute PageRank-based graph features (weighted + unweighted).

#     Unweighted PageRank: Every edge (Origin -> Destination) counts equally, and measures pure network centrality.
#     Weighted PageRank: Edges are weighted by the # of flights between airports. Airports with heavy traffic influence the graph more.
#     """

#     months = df.select("YEAR", "MONTH").distinct().collect()
#     results = []

#     for row in months:
#         y, m = row["YEAR"], row["MONTH"]

#         df_m = df.filter((col("YEAR") == y) & (col("MONTH") == m))

#         # 1. Build weighted edges
#         edges = (
#             df_m
#             .filter(col("ORIGIN").isNotNull() & col("DEST").isNotNull())
#             .groupBy("ORIGIN", "DEST")
#             .agg(F.count("*").alias("flight_count"))
#             .withColumnRenamed("ORIGIN", "src")
#             .withColumnRenamed("DEST", "dst")
#         )

#         # 2. Create vertices (origin + dest)
#         vertices = (
#             edges.select(col("src").alias("id"))
#             .union(edges.select(col("dst").alias("id")))
#             .distinct()
#         )


#         # 3. Unweighted PageRank
#         g_unweighted = GraphFrame(vertices, edges.select("src", "dst"))

#         pr_unweighted = g_unweighted.pageRank(
#             resetProbability=0.15,
#             maxIter=20
#         ).vertices.select(
#             col("id").alias("airport"),
#             col("pagerank").alias("pagerank_unweighted")
#         )

#         # 4. Weighted PageRank
#         edges_weighted = (
#             edges
#             .withColumn(
#                 "rep",
#                 F.sequence(F.lit(1), col("flight_count"))
#             )
#             .select("src", "dst", F.explode("rep"))
#             .select("src", "dst")
#         )

#         g_weighted = GraphFrame(vertices, edges_weighted)

#         pr_weighted = g_weighted.pageRank(
#             resetProbability=0.15,
#             maxIter=20
#         ).vertices.select(
#             col("id").alias("airport"),
#             col("pagerank").alias("pagerank_weighted")
#         )

#         # 5. Combine scores
#         pr = (
#             pr_unweighted
#             .join(pr_weighted, "airport", "outer")
#             .withColumn("YEAR", lit(y))
#             .withColumn("MONTH", lit(m))
#         )

#         results.append(pr)

#     return reduce(lambda a, b: a.unionByName(b), results)

In [0]:
train_edges

# Create Graph features - Unweighted and Weighted PageRank

In [0]:
def compute_unweighted_pagerank_features(otpw_df, edges, vertices):
    """
    Compute unweighted PageRank-based graph features.

    Unweighted PageRank: Every edge (Origin -> Destination) counts equally, and measures pure network centrality.
    """
    # Build unweighted graph
    unweighted_graph = GraphFrame(vertices, edges.select("src", "dst"))

    # Compute PageRank
    pagerank_unweighted_graph = unweighted_graph.pageRank(
        resetProbability=0.15,
        maxIter=20
    ).vertices.select(
        col("id").alias("AIRPORT"),
        col("pagerank").alias("unweighted_pagerank")
    )

    # Join ORIGIN
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_unweighted_graph.withColumnRenamed("AIRPORT", "ORIGIN")), 
            on="ORIGIN", 
            how="left"
        )
        .withColumnRenamed("unweighted_pagerank", "origin_pagerank_unweighted")
    )

    # Join DEST
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_unweighted_graph.withColumnRenamed("AIRPORT", "DEST")), 
            on="DEST", 
            how="left"
        )
        .withColumnRenamed("unweighted_pagerank", "dest_pagerank_unweighted")
    )

    return otpw_df
    

def compute_weighted_pagerank_features(otpw_df, edges, vertices):
    """
    Compute weighted PageRank-based graph features.

    Weighted PageRank: Edges are weighted by the # of flights between airports. Airports with heavy traffic influence the graph more.
    """
    # # Build weighted graph
    # weighted_graph = GraphFrame(vertices, edges)

    # # Compute PageRank
    # pagerank_weighted_graph = (
    #     weighted_graph.pageRank(
    #         resetProbability=0.15,
    #         maxIter=20,
    #         weightCol="WEIGHT"  
    #     ).vertices.select(
    #         col("id").alias("AIRPORT"),
    #         col("pagerank").alias("weighted_pagerank")
    #     )
    # )

    # Duplicate each edge according to its weight (i.e. weight = 500 means 500 copies of the edge between src-dst.)
    window = Window.partitionBy("src")
    edges_weighted = (
        edges
        .withColumn("seq", F.sequence(F.lit(0), col("weight").cast('int') - 1))
        .select("src", "dst", F.explode("seq"))
        .select("src", "dst")
    )


    # Build weighted graph
    weighted_graph = GraphFrame(vertices, edges_weighted)

    # Compute weighted PageRank
    pr = weighted_graph.pageRank(
        resetProbability=0.15,
        maxIter=20,
    )

    pagerank_weighted_graph = pr.vertices.select(
        col("id").alias("AIRPORT"),
        col("pagerank").alias("weighted_pagerank")
    )

    # Join ORIGIN
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_weighted_graph.withColumnRenamed("AIRPORT", "ORIGIN")), 
            on="ORIGIN", 
            how="left")
        .withColumnRenamed("weighted_pagerank", "origin_pagerank_weighted")
    )

    # Join DEST
    otpw_df = (
        otpw_df.join(
            broadcast(pagerank_weighted_graph.withColumnRenamed("AIRPORT", "DEST")), 
            on="DEST", 
            how="left")
        .withColumnRenamed("weighted_pagerank", "dest_pagerank_weighted")
    )

    return otpw_df

# Build edges and vertices for train and test sets

In [0]:
# Build weighted edges for otpw_train_df
train_edges = (
    otpw_train_df
    .filter(col("ORIGIN").isNotNull() & col("DEST").isNotNull())
    .groupBy("ORIGIN", "DEST")
    .agg(F.count("*").alias("FLIGHT_COUNT"))
    .withColumnRenamed("ORIGIN", "src")
    .withColumnRenamed("DEST", "dst")
    .withColumnRenamed('FLIGHT_COUNT', 'WEIGHT')
)

# 2. Create vertices (origin + dest)
train_vertices = (
    train_edges.select(col('src').alias("id"))
    .union(train_edges.select(col('dst').alias("id")))
    .distinct()
)

# Build weighted edges for otpw_test_df
test_edges = (
    otpw_test_df
    .filter(col("ORIGIN").isNotNull() & col("DEST").isNotNull())
    .groupBy("ORIGIN", "DEST")
    .agg(F.count("*").alias("FLIGHT_COUNT"))
    .withColumnRenamed("ORIGIN", "src")
    .withColumnRenamed("DEST", "dst")
    .withColumnRenamed('FLIGHT_COUNT', 'WEIGHT')
)

# 2. Create vertices (origin + dest)
test_vertices = (
    test_edges.select(col('src').alias("id"))
    .union(test_edges.select(col('dst').alias("id")))
    .distinct()
)

# Compute weighted and unweighted pagerank features for train and test sets

In [0]:
otpw_train_df = compute_weighted_pagerank_features(otpw_train_df, train_edges, train_vertices)
otpw_train_df = compute_unweighted_pagerank_features(otpw_train_df, train_edges, train_vertices)

otpw_test_df = compute_weighted_pagerank_features(otpw_test_df, test_edges, test_vertices)
otpw_test_df = compute_unweighted_pagerank_features(otpw_test_df, test_edges, test_vertices)

# Fill Missing Values

In [0]:
otpw_train_df = otpw_train_df.fillna({
    "origin_pagerank_weighted": 0.0,
    "origin_pagerank_unweighted": 0.0,
    "dest_pagerank_weighted": 0.0,
    "dest_pagerank_unweighted": 0.0
})

otpw_test_df = otpw_train_df.fillna({
    "origin_pagerank_weighted": 0.0,
    "origin_pagerank_unweighted": 0.0,
    "dest_pagerank_weighted": 0.0,
    "dest_pagerank_unweighted": 0.0
})

# Write otpw train and test sets with window-based and PageRank features

In [0]:
otpw_train_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041126_otpw_train_df.parquet")
otpw_test_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041126_otpw_test_df.parquet")

In [0]:
display(otpw_train_df)

In [0]:
display(dbutils.fs.ls(TEAM_DIR))

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
#df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
otpw_full_df = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")
otpw_full_df.printSchema()

### Summary from Phase II (still need to revise for Phase III)

Focused on the 5 most important weather variables:
* Precipitation (PrecipBinary): 1 if rain, 0 if no rain
  * this was heavily skewed so i decided to make it binary
* Humidity (HourlyRelativeHumidity_scaled): nomalized (mean 0, std 1)
* Temperature (HourlyDryBulbTemperature_scaled): nomalized (mean 0, std 1)
* Wind Speed (HourlyWindSpeed_scaled): log+1 transformed and nomalized (mean 0, std 1)
* Visibility (VisibilityCat):
| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

I also created additional features such as IsWeekend and Route. Route combines origin and destination and is one hot encoded for ML modeling.

The notebook also includes steps taken to filter out cancelled fights and to only use FM-15 data.

# Future Plans

TODO:
* Specify the feature transformations for the pipeline and justify these features given the target (e.g., hashing trick, tf-idf, stopword removal, lemmatization, tokenization, etc..)
* Other feature engineering efforts, i.e., interaction terms, Breiman's method, etc.

We are concerned with false negatives. This is the case where we predict dep_delay15 is 0 (no delay) when there is actually a delay (1). Therefore we will concern ourselves with measuring recall. We will avoid accuracy since the dataset is imbalanced; there are a lot more non-delayed flights then there are delayed ones.